In [1]:
import pandas as pd
import numpy as np

In [4]:
nav = pd.read_csv("./data/raw/02_nav_history.csv")

print(nav.shape)
print(nav.head())
print(nav.isnull().sum())

(46000, 3)
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
amfi_code    0
date         0
nav          0
dtype: int64


In [6]:
# Convert date column

nav['date'] = pd.to_datetime(nav['date'])

# Sort by fund and date

nav = nav.sort_values(
    ['amfi_code','date']
)

# Remove duplicates

nav = nav.drop_duplicates()

# Remove invalid NAV values

nav = nav[nav['nav'] > 0]

# Forward fill missing NAV

nav['nav'] = (
    nav.groupby('amfi_code')['nav']
       .ffill()
)

nav.to_csv(
    "./data/processed/02_nav_history_clean.csv",
    index=False
)

print(nav.shape)

(46000, 3)


In [7]:
txn = pd.read_csv(
    "./data/raw/08_investor_transactions.csv"
)

print(txn.shape)
txn.head()

(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [11]:
txn['transaction_date'] = pd.to_datetime(
    txn['transaction_date']
)

txn['transaction_type'] = (
    txn['transaction_type']
    .str.strip()
    .str.lower()
)

mapping = {
    'sip':'SIP',
    'lumpsum':'Lumpsum',
    'redemption':'Redemption'
}

txn['transaction_type'] = (
    txn['transaction_type']
    .replace(mapping)
)

txn = txn[txn['amount_inr'] > 0]

valid_kyc = [
    'Verified',
    'Pending',
    'Rejected'
]

txn = txn[
    txn['kyc_status'].isin(valid_kyc)
]

txn.to_csv(
    "./data/processed/08_investor_transactions_clean.csv",
    index=False
)

print(txn.shape)

(32778, 13)


In [13]:
perf = pd.read_csv(
    "./data/raw/07_scheme_performance.csv"
)

perf.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [16]:
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
]

for col in return_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors='coerce'
    )

perf['expense_ratio_flag'] = (
    (perf['expense_ratio_pct'] < 0.1)
    |
    (perf['expense_ratio_pct'] > 2.5)
)

perf.to_csv(
    "./data/processed/07_scheme_performance_clean.csv",
    index=False
)

perf.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,expense_ratio_flag
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,False
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,False
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,False
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,False
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,False


In [19]:
from sqlalchemy import create_engine
import os

engine = create_engine("sqlite:///db/bluestock_mf.db")

print("DB Path:", os.path.abspath("db/bluestock_mf.db"))

DB Path: c:\Users\vamsi\OneDrive\Desktop\mutual-fund-analytics\db\bluestock_mf.db


In [20]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///db/bluestock_mf.db")

with engine.connect() as conn:
    print("Database connected successfully")

Database connected successfully


In [21]:
import os

print(os.path.exists("db/bluestock_mf.db"))

True


In [22]:
import pandas as pd

fund = pd.read_csv("./data/raw/01_fund_master.csv")

# Remove duplicates
fund = fund.drop_duplicates()

# Remove leading/trailing spaces from text columns
for col in fund.select_dtypes(include='object').columns:
    fund[col] = fund[col].str.strip()

# Handle missing values
fund = fund.fillna("Unknown")

fund.to_csv(
    "./data/processed/01_fund_master_clean.csv",
    index=False
)

print(fund.shape)

(40, 15)


In [23]:
import pandas as pd

aum = pd.read_csv("./data/raw/03_aum_by_fund_house.csv")

aum = aum.drop_duplicates()

# Convert AUM to numeric
aum['aum_crore'] = pd.to_numeric(
    aum['aum_crore'],
    errors='coerce'
)

# Keep positive AUM only
aum = aum[aum['aum_crore'] > 0]

aum.to_csv(
    "./data/processed/03_aum_by_fund_house_clean.csv",
    index=False
)

print(aum.shape)

(90, 5)


In [24]:
import pandas as pd

sip = pd.read_csv("./data/raw/04_monthly_sip_inflows.csv")

sip = sip.drop_duplicates()

# Convert month column
sip['month'] = pd.to_datetime(
    sip['month'],
    errors='coerce'
)

# Validate SIP amount
sip['sip_inflow_crore'] = pd.to_numeric(
    sip['sip_inflow_crore'],
    errors='coerce'
)

sip = sip[sip['sip_inflow_crore'] > 0]

sip.to_csv(
    "./data/processed/04_monthly_sip_inflows_clean.csv",
    index=False
)

print(sip.shape)

(48, 6)


In [25]:
import pandas as pd

cat = pd.read_csv("./data/raw/05_category_inflows.csv")

cat = cat.drop_duplicates()

# Numeric validation
cat['net_inflow_crore'] = pd.to_numeric(
    cat['net_inflow_crore'],
    errors='coerce'
)

cat.to_csv(
    "./data/processed/05_category_inflows_clean.csv",
    index=False
)

print(cat.shape)

(144, 3)


In [27]:
import pandas as pd

folio = pd.read_csv(
    "./data/raw/06_industry_folio_count.csv"
)

# Convert month to datetime
folio['month'] = pd.to_datetime(
    folio['month'],
    errors='coerce'
)

# Remove duplicates
folio = folio.drop_duplicates()

# Convert folio columns to numeric

folio_cols = [
    'total_folios_crore',
    'equity_folios_crore',
    'debt_folios_crore',
    'hybrid_folios_crore',
    'others_folios_crore'
]

for col in folio_cols:
    folio[col] = pd.to_numeric(
        folio[col],
        errors='coerce'
    )

# Validate values are non-negative

for col in folio_cols:
    folio = folio[folio[col] >= 0]

# Save cleaned file

folio.to_csv(
    "./data/processed/06_industry_folio_count_clean.csv",
    index=False
)

print("Shape:", folio.shape)
print(folio.head())

Shape: (21, 6)
       month  total_folios_crore  equity_folios_crore  debt_folios_crore  \
0 2022-01-01               13.26                 9.28               1.86   
1 2022-04-01               13.91                 9.74               1.95   
2 2022-07-01               13.85                 9.69               1.94   
3 2022-10-01               14.12                 9.88               1.98   
4 2023-01-01               14.81                10.37               2.07   

   hybrid_folios_crore  others_folios_crore  
0                 0.80                 1.33  
1                 0.83                 1.39  
2                 0.83                 1.38  
3                 0.85                 1.41  
4                 0.89                 1.48  


In [30]:
import pandas as pd

holdings = pd.read_csv(
    "./data/raw/09_portfolio_holdings.csv"
)

# Convert portfolio date
holdings['portfolio_date'] = pd.to_datetime(
    holdings['portfolio_date'],
    errors='coerce'
)

# Remove duplicates
holdings = holdings.drop_duplicates()

# Convert numeric columns

numeric_cols = [
    'weight_pct',
    'market_value_cr',
    'current_price_inr'
]

for col in numeric_cols:
    holdings[col] = pd.to_numeric(
        holdings[col],
        errors='coerce'
    )

# Validate values

holdings = holdings[
    (holdings['weight_pct'] >= 0) &
    (holdings['weight_pct'] <= 100)
]

holdings = holdings[
    holdings['market_value_cr'] >= 0
]

holdings = holdings[
    holdings['current_price_inr'] > 0
]

# Remove rows with missing key fields

holdings = holdings.dropna(
    subset=[
        'amfi_code',
        'stock_symbol',
        'stock_name',
        'portfolio_date'
    ]
)

# Save cleaned file

holdings.to_csv(
    "./data/processed/09_portfolio_holdings_clean.csv",
    index=False
)

print("Shape:", holdings.shape)
print(holdings.head())

Shape: (322, 8)
   amfi_code stock_symbol                stock_name       sector  weight_pct  \
0     119551    POWERGRID    Power Grid Corporation    Utilities       13.85   
1     119551     HDFCBANK             HDFC Bank Ltd      Banking       11.19   
2     119551       GRASIM     Grasim Industries Ltd  Diversified        9.90   
3     119551      DRREDDY  Dr. Reddy's Laboratories       Pharma        4.76   
4     119551   ASIANPAINT          Asian Paints Ltd       Paints       10.25   

   market_value_cr  current_price_inr portfolio_date  
0           737.09            6011.08     2025-12-31  
1            88.97            1074.65     2025-12-31  
2           208.45            5964.59     2025-12-31  
3           161.32            3748.82     2025-12-31  
4           725.90            1321.45     2025-12-31  


In [29]:
import pandas as pd

benchmark = pd.read_csv(
    "./data/raw/10_benchmark_indices.csv"
)

benchmark = benchmark.drop_duplicates()

# Date conversion
benchmark['date'] = pd.to_datetime(
    benchmark['date'],
    errors='coerce'
)

benchmark.to_csv(
    "./data/processed/10_benchmark_indices_clean.csv",
    index=False
)

print(benchmark.shape)

(8050, 3)
